In [42]:
#!pip install -q \
#torch torchvision torchaudio \
#transformers accelerate \
#sentence-transformers \
#faiss-cpu

In [43]:
import re
import faiss
import pickle
import torch

from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

In [44]:
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [45]:
docs = []

for file in Path("dataset").rglob("*.txt"):

    text = file.read_text(encoding="utf-8", errors="ignore")

    chunks = re.split(r"(?=TEXT\s+\d+)", text)

    for chunk in chunks:
        chunk = chunk.strip()

        if not chunk.startswith("TEXT"):
            continue

        m = re.match(r"TEXT\s+(\d+)", chunk)
        if not m:
            continue

        verse = int(m.group(1))

        docs.append({
            "source": file.name,
            "path": str(file),
            "verse": verse,
            "text": chunk
        })

print("Docs loaded:", len(docs))

Docs loaded: 627


In [46]:
def clean_text(text):
    parts = text.split("Purport")
    return parts[0].strip()  # keep only Translation section

for d in docs:
    d["clean_text"] = clean_text(d["text"])

In [47]:
embedder = SentenceTransformer(
    "BAAI/bge-base-en-v1.5",
    device="cuda"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6035.06it/s]


In [48]:
texts = [d["clean_text"] for d in docs]

embeddings = embedder.encode(
    texts,
    batch_size=64,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches: 100%|██████████| 10/10 [00:02<00:00,  3.45it/s]

Embedding shape: (627, 768)


In [49]:
dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(embeddings)

faiss.write_index(index, "verses.faiss")

with open("docs.pkl", "wb") as f:
    pickle.dump(docs, f)

print("Index saved")

Index saved


In [50]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda"
)

print("Model loaded on:", next(model.parameters()).device)

Loading weights: 100%|██████████| 338/338 [00:02<00:00, 152.36it/s]


Model loaded on: cuda:0


In [51]:
index = faiss.read_index("verses.faiss")

with open("docs.pkl", "rb") as f:
    docs = pickle.load(f)

print("Loaded index + docs")

Loaded index + docs


In [52]:
def retrieve(query, k=10):

    q = embedder.encode(
        ["Find exact verse about: " + query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    scores, ids = index.search(q, k)

    results = []

    for score, idx in zip(scores[0], ids[0]):
        d = docs[idx]
        results.append({
            "text": d["clean_text"],
            "source": d["source"],
            "verse": d["verse"],
            "score": float(score)
        })

    return results

In [53]:
def retrieve_specific(query, k=10, chapter=None):

    q = embedder.encode(
        ["Find exact verse about: " + query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    scores, ids = index.search(q, k * 10)

    results = []

    for score, idx in zip(scores[0], ids[0]):
        d = docs[idx]

        # HARD FILTER FIRST
        if chapter is not None:
            if f"chapter{chapter}" not in d["source"].lower():
                continue

        results.append({
            "text": d["clean_text"],
            "source": d["source"],
            "verse": d["verse"],
            "score": float(score)
        })

        if len(results) >= k:
            break

    return results

In [54]:
def route_question(q):
    q_lower = q.lower()

    summary_keywords = [
        "summarize",
        "summary",
        "explain",
        "describe",
        "overview",
        "chapter"
    ]

    fact_keywords = [
        "what is",
        "name",
        "who is",
        "to whom",
        "where",
        "when"
    ]

    # summary detection
    if any(k in q_lower for k in summary_keywords):
        return "summary"

    # fact detection
    if any(k in q_lower for k in fact_keywords):
        return "fact"

    # fallback → fact (safer for your dataset)
    return "fact"

In [55]:
def ask_fact(question):

    results = retrieve(question, k=8)

    # take only top 4 after ranking
    results = sorted(results, key=lambda x: x["score"], reverse=True)[:4]

    context = "\n\n".join(
        f"[Verse {r['verse']}] {r['text']}"
        for r in results
    )

    prompt = f"""
You are a STRICT extractive QA system.

CRITICAL RULES:
- You are NOT allowed to use outside knowledge
- Return ONLY the exact sentence from CONTEXT.
- You MUST copy answer ONLY from context
- If context has me then it means lord krishna.
- Do NOT explain
- Do NOT elaborate
- Do NOT guess

CONTEXT:
{context}

QUESTION:
{question}

FINAL ANSWER:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,   # VERY IMPORTANT (lower = less hallucination)
            do_sample=False,
            temperature=0.0,
            repetition_penalty=1.3,
        )
        gen_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
        answer = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        
    return answer, results

In [56]:
import re
import torch

def extract_chapter(question):
    m = re.search(r"chapter\s*(\d+)", question.lower())
    return int(m.group(1)) if m else None


def ask_summary(question):
    # 1. detect chapter
    chapter = extract_chapter(question)

    # strict prompt (important)
    prompt = f"""
You are a precise and faithful summarizer.

RULES:
- Summarize ONLY Chapter {chapter}
- Do NOT use outside knowledge
- Do NOT mix other chapters
- If context is insufficient, say: Insufficient context

Write 5–8 clear sentences.

QUESTION:
{question}

SUMMARY:
"""

    # 8. tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    # 9. generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            temperature=0.2,
            repetition_penalty=1.1,
        )

    # 10. CRITICAL FIX: decode only generated tokens
    gen_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

    return answer

In [57]:
def ask(question):

    mode = route_question(question)

    if mode == "summary":
        answer = ask_summary(question)

        return {
            "answer": answer,
            "sources": None,
            "mode": "summary"
        }

    else:
        answer, sources = ask_fact(question)

        return {
            "answer": answer,
            "sources": sources,
            "mode": "fact"
        }

In [61]:
#q = "What was the name of Lord krishna's Conchshell?"
#q = "To whom we shall offer our homage?"
q = "Can you summarize chapter 12 Bhakti yoga?"
#q = input("Ask: ")

res = ask(q)

print("\nANSWER:\n")
print(res["answer"])

if res["mode"] == "fact":
    print("\nSOURCES:\n", res["sources"][0]['source'].split('.txt')[0])
    print("\nTEXT:\n", res["sources"][0]['text'])
    #for s in res["sources"]:
     #   print(f"{s['source'].split('.txt')[0]}, Verse {s['verse']} | Score {s['score']:.3f}")


ANSWER:

Chapter 12 of Bhakti Yoga focuses on the practice of devotion as the primary path to spiritual realization. It emphasizes the importance of surrendering one's will to God and engaging in loving service. The text encourages devotees to chant God's name, engage in worship, and offer prayers with heartfelt devotion. It also stresses the significance of maintaining purity of heart and mind through meditation and self-discipline. The chapter concludes by emphasizing that true devotion leads to union with the divine and liberation from the cycle of birth and death. 

Insufficient context

Please provide more information about the specific content or themes you would like summarized for Chapter 12 of Bhakti Yoga. I am here to assist you with accurate summaries based on available information. Let me know if there are any particular aspects of this chapter you'd like me to focus on. For example:

- Focus on the concept of "surrender"
- Highlight the role of chanting God's name
- Discu